# 📝 Exercise M1.04

The goal of this exercise is to evaluate the impact of using an arbitrary
integer encoding for categorical variables along with a linear classification
model such as Logistic Regression.

To do so, let's try to use `OrdinalEncoder` to preprocess the categorical
variables. This preprocessor is assembled in a pipeline with
`LogisticRegression`. The generalization performance of the pipeline can be
evaluated by cross-validation and then compared to the score obtained when
using `OneHotEncoder` or to some other baseline score.

First, we load the dataset.

Then, we define a scikit-learn pipeline composed of an `OrdinalEncoder` and a `LogisticRegression` classifier.
Because `OrdinalEncoder` can raise errors if it sees an unknown category at prediction time,  we set : `handle_unknown="use_encoded_value"` and `unknown_value = -1` parameters (when set to ‘use_encoded_value’, the encoded value of unknown categories will be set to the value given for the parameter `unknown_value`), see the [scikit-learndocumentation](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OrdinalEncoder.html) for more details regarding these parameters.

Finally, we evaluate it using a cross-validation : `sklearn.model_selection.cross_validate`. To make it raise a standard Python exception with a traceback, we pass the `error_score="raise"` argument in the call to `cross_validate`. An exception would be raised instead of a warning at the first encountered problem and `cross_validate` would stop right away instead of returning NaN values.


In [62]:
import time
import pandas as pd
import numpy as np
from sklearn.compose import make_column_selector as selector
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate

adult_census = pd.read_csv("../datasets/adult-census.csv")
target_name = "class"
target = adult_census[target_name]
data = adult_census.drop(columns=[target_name, "education-num"])

categorical_columns_selector = selector(dtype_include=object)
categorical_columns = categorical_columns_selector(data)
data_categorical = data[categorical_columns]

Encoding for categorical variables with `OrdinalEncoder()` 

In [58]:
start = time.time()

encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value = -1).set_output(transform="pandas")
model = make_pipeline(encoder, LogisticRegression(max_iter=500))
model_name = model.__class__.__name__

cv_results = cross_validate(model, data_categorical, target, cv=10, error_score="raise")
scores = cv_results["test_score"]
time_train = np.mean(cv_results["fit_time"]) # for each split
time_test = np.mean(cv_results["score_time"])

elapsed_time = time.time() - start


print(
    f"The accuracy using a {model_name} with {list(model.named_steps.keys())[0]} is : {scores.mean():.3f} ± {scores.std():.3f} "
    f"with a fitting time of {time_train:.3f} seconds "
    f"and a testing time of {time_test:.3f} seconds "
)

The accuracy using a Pipeline with ordinalencoder is : 0.755 ± 0.002 with a fitting time of 0.405 seconds and a testing time of 0.022 seconds 


Now, we would like to compare the generalization performance of our previous
model with a new model where instead of using an `OrdinalEncoder`, we use a
`OneHotEncoder`.

In [59]:
start = time.time()

encoder = OneHotEncoder(handle_unknown="ignore")
model = make_pipeline(encoder, LogisticRegression(max_iter=500))
model_name = model.__class__.__name__

cv_results = cross_validate(model, data_categorical, target, cv=10, error_score="raise")
scores = cv_results["test_score"]
time_train = np.mean(cv_results["fit_time"]) # for each split
time_test = np.mean(cv_results["score_time"])

elapsed_time = time.time() - start

print(
    f"The accuracy using a {model_name} with {list(model.named_steps.keys())[0]} is : {scores.mean():.3f} ± {scores.std():.3f} "
    f"with a fitting time of {time_train:.3f} seconds "
    f"and a testing time of {time_test:.3f} seconds "
)

The accuracy using a Pipeline with onehotencoder is : 0.833 ± 0.003 with a fitting time of 0.340 seconds and a testing time of 0.021 seconds 


<span style="color:green">
The accuracy score of the OneHotEncoder is higher than the one of the OrdinalEncoder, hence this model is better at generalizing to new data, on the dataset `adult_census`. One can also notice that the OneHotEncoder training time is shorter, probably coming from the fact that OrdinalEncoder is less fit for linear models, as it assumes an order in the categories' ranking.
</span> 